# CFPB complaint classifier: DistilBERT on Colab (CSCI 544, Group 36)

Fine-tunes `distilbert-base-uncased` to predict the complaint **product** (11 classes) and, optionally, the **issue** (89 classes) from the cleaned narrative. It reports per-class F1, macro-F1, and a confusion matrix.

**Before you start (one time):**
1. **GPU:** *Runtime → Change runtime type → T4 GPU* (or better).
2. **GitHub token** (the repo is private): Aditya adds you as a collaborator, and you accept the invite. Then on GitHub go to *Settings → Developer settings → Personal access tokens → **Tokens (classic)** → Generate new token*, tick the **`repo`** scope, and set a short expiry (for example 30 days). Fine-grained tokens cannot reach a repo owned by another personal account. In Colab, click the **🔑 Secrets** icon in the left sidebar, add a secret named `GITHUB_TOKEN` with the token as its value, and switch on *Notebook access*. Never paste the token into a cell.
3. **Data (fastest):** ask Aditya to share a Google Drive folder named `cfpb-grounded-rag` containing `data/processed/train.parquet`, `val.parquet`, `test.parquet`, and `annotation_sample.csv`, about 220 MB. Add a shortcut to it in *My Drive*. Without it, the notebook rebuilds the data from the CFPB archive, which takes about 15 extra minutes and about 8 GB of RAM.

Then run the cells **top to bottom**. Everything important is saved to Google Drive, so a disconnect loses at most one epoch.

## 1. Check the GPU

In [ ]:
import torch, subprocess
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("NO GPU: Runtime > Change runtime type > T4 GPU, then Runtime > Restart and run again.")

## 2. Mount Google Drive (results and checkpoints are saved here)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/cfpb-grounded-rag'   # change if your shared folder lives elsewhere
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive folder:', DRIVE_DIR, '| contents:', os.listdir(DRIVE_DIR))

## 3. Get the code (private repo, token read from Colab Secrets)

In [ ]:
import subprocess, os
from google.colab import userdata

BRANCH = 'classifier'
REPO = '/content/cfpb-grounded-rag'
token = userdata.get('GITHUB_TOKEN')
url = f'https://{token}@github.com/Aditya-Datar/cfpb-grounded-rag.git'

def run(cmd, cwd=None):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(token, '***'))   # never echo the token
    if r.returncode != 0:
        raise RuntimeError('command failed (see message above)')

if not os.path.isdir(REPO):
    run(['git', 'clone', '--quiet', '--branch', BRANCH, url, REPO])
else:
    run(['git', 'remote', 'set-url', 'origin', url], cwd=REPO)
    run(['git', 'pull', '--quiet', 'origin', BRANCH], cwd=REPO)
# Remove the token from the saved remote URL.
run(['git', 'remote', 'set-url', 'origin', 'https://github.com/Aditya-Datar/cfpb-grounded-rag.git'], cwd=REPO)
run(['git', 'log', '--oneline', '-1'], cwd=REPO)
os.chdir(REPO)
print('Working directory:', os.getcwd())

## 4. Install dependencies (Colab already has torch and transformers; this only fills gaps)

In [ ]:
!pip install -q -r requirements-classifier.txt
import transformers, pandas, sklearn
print('transformers', transformers.__version__, '| pandas', pandas.__version__, '| sklearn', sklearn.__version__)

## 5. Get the data splits

Uses the shared Drive copy if present (seconds). Otherwise it downloads the CFPB FOIA archive (about 365 MB, cached on Drive) and rebuilds the splits (about 15 minutes, about 8 GB RAM). Either way, `check_splits.py` confirms you have **exactly the team's rows**.

In [ ]:
import os, shutil, subprocess
SRC = f'{DRIVE_DIR}/data/processed'
DST = 'data/processed'
need = ['train.parquet', 'val.parquet', 'test.parquet', 'annotation_sample.csv']
os.makedirs(DST, exist_ok=True)

if all(os.path.exists(f'{SRC}/{f}') for f in need):
    for f in need:
        shutil.copy(f'{SRC}/{f}', f'{DST}/{f}')
    print('Copied splits from Drive.')
else:
    print('Splits not on Drive: rebuilding from the CFPB archive (about 15 min)...')
    archive_drive = f'{DRIVE_DIR}/data/raw/foia_archive'
    os.makedirs(archive_drive, exist_ok=True)
    os.makedirs('data/raw', exist_ok=True)
    if not os.path.exists('data/raw/foia_archive'):
        os.symlink(archive_drive, 'data/raw/foia_archive')   # cache the zips on Drive for next time
    get_ipython().system('python scripts/fetch_narrative_archive.py')
    get_ipython().system('python scripts/run_data_pipeline.py')
    os.makedirs(SRC, exist_ok=True)
    for f in need:
        shutil.copy(f'{DST}/{f}', f'{SRC}/{f}')
    print('Saved splits to Drive for next time.')

!python scripts/check_splits.py

## 6. Smoke test (about 2 to 5 minutes)

A tiny run that checks the whole pipeline works on this GPU before spending an hour. It writes to `/content/smoke`, not Drive. Ignore its scores, which are meaningless at this size.

In [ ]:
!python scripts/train_classifier.py --label product --max-train 2000 --max-eval 500 --epochs 1 --log-every 20 --out /content/smoke

## 7. Full run: **product** (11 classes)

Defaults: 3 epochs, batch size 32, lr 5e-5, max 512 tokens with head+tail truncation, keeping the best epoch on validation. The log prints speed and **epoch ETA**; GPU timing has not been measured yet, so note it for the team.

- **Disconnected?** Rerun cells 1 to 5, then rerun this cell unchanged. `--resume` continues from the last finished epoch saved on Drive.
- **`CUDA out of memory`?** Add `--batch-size 16`. If the run is a fresh start, delete the Drive output folder first.

In [ ]:
OUT_PRODUCT = f'{DRIVE_DIR}/outputs/classifier/product'
!python scripts/train_classifier.py --label product --out "{OUT_PRODUCT}" --resume

## 8. Results: product

In [ ]:
import json, pandas as pd
from IPython.display import Image, display

def show(out_dir):
    info = json.load(open(f'{out_dir}/run_info.json'))
    print('GPU:', info['gpu'], '| epochs:', [(h['epoch'], round(h['val_macro_f1'], 4), round(h['epoch_minutes'], 1)) for h in info['history']])
    for split, r in info['results'].items():
        print(f"{split}: macro-F1 {r['macro_f1']:.4f} | accuracy {r['accuracy']:.4f} | weighted-F1 {r['weighted_f1']:.4f} | n={r['n']} (excluded unseen labels: {r['n_excluded_unseen_label']})")
    rep = pd.read_csv(f'{out_dir}/classification_report_test.csv', index_col=0)
    display(rep.round(3))
    display(Image(f'{out_dir}/confusion_matrix_test.png'))

show(OUT_PRODUCT)

## 9. (Optional) Full run: **issue** (89 classes)

Same procedure; it takes about as long as the product run. Issues that appear in val/test but never in train cannot be predicted, so they are excluded from metrics and counted in `n_excluded_unseen_label`.

In [ ]:
OUT_ISSUE = f'{DRIVE_DIR}/outputs/classifier/issue'
!python scripts/train_classifier.py --label issue --out "{OUT_ISSUE}" --resume
show(OUT_ISSUE)

## 10. Report back to the team

Everything is in `MyDrive/cfpb-grounded-rag/outputs/classifier/<label>/`:

| File | What it is |
|---|---|
| `run_info.json` | settings, per-epoch history, val/test scores, GPU, library versions |
| `metrics_test.json`, `classification_report_test.csv` | per-class precision, recall, F1, plus macro-F1 |
| `confusion_matrix_test.png` / `.csv` | which products get confused |
| `predictions_test.parquet` | `complaint_id`, true label, predicted label, probability (the stage interface) |
| `best/` | the best model, reusable for the Streamlit demo |

Send `run_info.json` and the test report to whoever updates `CLAUDE_PROGRESS.md`, and add a dated entry to its Sec. 7 with the GPU, minutes per epoch, and test macro-F1. **Do not tune hyperparameters on the test set:** compare settings on validation only.